In [1]:
import math
import pandas as pd
import json
from alive_progress import alive_bar
print("Finished imports.")

Finished imports.


In [2]:
raw_df = pd.read_csv('../data/edges_saigon_elevation.csv')

In [3]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points on the Earth in meters.
    """
    R = 6371000  # Earth radius in meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lambda = math.radians(lon2 - lon1)

    a = math.sin(d_phi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lambda / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

In [4]:

def calculate_slope(lat1, lon1, ele1, lat2, lon2, ele2):
    """
    Calculate the slope between two geographic points.

    Returns:
        slope_ratio (float): Rise over run.
        slope_percent (float): Slope as a percentage.
        slope_angle_deg (float): Slope in degrees.
    """
    horizontal_distance = haversine(lat1, lon1, lat2, lon2)
    vertical_rise = ele2 - ele1

    if horizontal_distance == 0:
        return float('inf'), float('inf'), 90.0 if vertical_rise != 0 else 0.0

    slope_ratio = vertical_rise / horizontal_distance
    # slope_percent = slope_ratio * 100
    # slope_angle_deg = math.degrees(math.atan(slope_ratio))

    return abs(slope_ratio)#, slope_percent, slope_angle_deg


In [5]:
def parse_linestring(wkt_str):
    """
    Parses a WKT LINESTRING or MULTILINESTRING and returns a flat list of (lat, lon) tuples.

    Parameters:
        wkt_str (str): A WKT string in LINESTRING or MULTILINESTRING format.

    Returns:
        List[Tuple[float, float]]: List of (lat, lon) tuples.
    """
    wkt_str = wkt_str.strip()
    
    if wkt_str.startswith("LINESTRING"):
        coord_part = wkt_str[len("LINESTRING"):].strip(" ()")
        segments = [coord_part]
        
    elif wkt_str.startswith("MULTILINESTRING"):
        coord_part = wkt_str[len("MULTILINESTRING"):].strip(" ()")
        segments = coord_part.split("), (")
        
    else:
        raise ValueError("Input must start with 'LINESTRING' or 'MULTILINESTRING'")

    latlon_list = []

    for segment in segments:
        coords = segment.replace("(", "").replace(")", "").split(",")
        for pair in coords:
            lon_str, lat_str = pair.strip().split()
            latlon_list.append((float(lat_str), float(lon_str)))

    return latlon_list

In [6]:
metrics = []  # Will store slope and elevation metrics for each linestring segment

# Use a progress bar to visualize progress through raw_df
with alive_bar(len(raw_df), force_tty=True) as bar:
    try:
        for index, row in raw_df.iterrows():
            if row['elevation_m_lmsl'] == "na":
                metrics.append({})
                continue
            row_metrics = {}  # Dictionary to hold metrics for the current row
            row_metrics["all_slope_ratios"] = []  # Will store slope ratios between each pair of points

            # Parse geometry string into list of (lon, lat) coordinate tuples
            point_list = parse_linestring(row["geometry"])

            # Load list of elevation values corresponding to each point
            elevations_list = json.loads(row["elevation_m_lmsl"])

            # Compute slope between each pair of consecutive points
            for i in range(len(point_list) - 1):
                slope_value = calculate_slope(
                    point_list[i][0], point_list[i][1], elevations_list[i],       # Start point and elevation
                    point_list[i+1][0], point_list[i+1][1], elevations_list[i+1]  # End point and elevation
                )

                # Some slope functions may return a tuple (e.g., if error/info is included)
                # Default to 0.0 if slope_value is not a simple float
                row_metrics["all_slope_ratios"].append(
                    slope_value if not isinstance(slope_value, tuple) else 0.0
                )

            # Store maximum absolute slope encountered along the segment
            row_metrics["abs_max_slope"] = max(row_metrics["all_slope_ratios"])

            # Compute absolute elevation difference from start to end of segment
            row_metrics["abs_elevation_diff"] = abs(elevations_list[-1] - elevations_list[0])

            # Compute total elevation range (max - min) over the whole segment
            row_metrics["max_abs_elevation_diff"] = max(elevations_list) - min(elevations_list)

            metrics.append(row_metrics)  # Save the metrics for this segment
            bar()  # Update progress bar

    except Exception as e:
        # Print error and current slope ratios for debugging
        print("Error:", e)
        print(row_metrics["all_slope_ratios"])

|████████████████████████████████████████| 2456/2456 [100%] in 0.3s (7712.78/s) 


In [7]:
raw_df['slope_metrics'] = metrics
raw_df.to_csv('../data/edges_saigon_elevation', index=False)